# Bearbeitungsplan

Ziel der Bearbeitung ist die Lösung des Bin-Packing-Problems mit einer zusätzlichen konstruktiven Heuristik und einer Metaheuristik. Als Metaheuristik wird **Simulated Annealing** verwendet. Das Notebook dokumentiert den Lösungsweg, sammelt die Parameterentscheidungen und gibt die Ergebnisse für alle Datensätze aus.

## Vorgehen

1. Problem und Daten prüfen.
2. Konstruktive Startlösung erzeugen und zusätzliche Startheuristik ergänzen.
3. Bewertungslogik und Zulässigkeitsprüfung verwenden.
4. Simulated Annealing implementieren und parametrisieren.
5. Alle Instanzen lösen, prüfen und als CSV speichern.
6. Dokumentation mit Klassifizierung, Suchoperatoren, Diversifizierung, Intensivierung, Terminierungskriterium, Parametern und Schwierigkeiten finalisieren.


# 1. Imports und globale Einstellungen

In [1]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations

import csv
import os
import time

import numpy as np

from Solver import *

In [ ]:
START_HEURISTIC = 'BPI'

SA_PARAMETERS = {
    'neighborhoodEvaluationStrategy': 'FirstImprovement',
    'neighborhoodTypes': ['Swap'],
    'temperature': 0.9,
    'coolingSpeed': 0.75,
}

seed = 2


# 2. Daten laden

In [4]:
# Die Klasse Files liegt in InputData.py und sucht sortierte JSON-Dateien im Ordner ../Data.
# InputData.py enthält außerdem InputData, DataItem und DataBinCapacity.

files = Files()

try:
    paths = sorted(files.GetFiles())
except FileNotFoundError:
    paths = []
    print("Kein Data-Ordner gefunden. Lege ../Data an oder passe InputData.Files an.")

fileNames = [path.split('/')[-1] for path in paths]
print(f"Alle Dateien im Zielordner sind: {fileNames} \n")

dataSets = []
for path in paths:
    print("________________________________________________________________________________________")
    print(f"Lade Instanz: {path.split('/')[-1]}")

    # InputData.DataLoad validiert die JSON-Datei und baut DataItem-Objekte und DataBinCapacity auf.
    data = InputData(path)
    dataSets.append(data)

Alle Dateien im Zielordner sind: ['..\\Data\\Falkenauer_u1000_13.json', '..\\Data\\Falkenauer_u120_09.json', '..\\Data\\Falkenauer_u500_05.json', '..\\Data\\csBA500_12.json', '..\\Data\\csBB250_13.json'] 

________________________________________________________________________________________
Lade Instanz: ..\Data\Falkenauer_u1000_13.json
Number of items: 1000
Bincapacity: 150

________________________________________________________________________________________
Lade Instanz: ..\Data\Falkenauer_u120_09.json
Number of items: 120
Bincapacity: 150

________________________________________________________________________________________
Lade Instanz: ..\Data\Falkenauer_u500_05.json
Number of items: 500
Bincapacity: 150

________________________________________________________________________________________
Lade Instanz: ..\Data\csBA500_12.json
Number of items: 5220
Bincapacity: 1500000

________________________________________________________________________________________
Lade Insta

# 3. Konstruktive Startlösungen

In [ ]:
constructiveResults = []
#Changed for faster testing: remove this line:
dataSets = [dataSets[0]]
#########################
for data in dataSets:
    print("________________________________________________________________________________________")
    print(f"Konstruktive Phase für: {data.filename}")

    startTime = time.time()
    
    # Solver liegt in Solver.py und koordiniert InputData, EvaluationLogic, SolutionPool und Heuristiken.
    # Die eigentliche Startheuristik liegt in ConstructiveHeuristics.py.
    solver = Solver(data)
    startSolution = solver.ConstructionPhase(START_HEURISTIC)

    # Solution und FeasibilityCheck liegen in OutputData.py.
    startSolution.FeasibilityCheck(data)
    print(startSolution.Bins)

    runtime = time.time() - startTime
    constructiveResults.append({
        'path': data.path,
        'instance': data.filename,
        'data': data,
        'heuristic': START_HEURISTIC,
        'startSolution': startSolution,
        'constructiveRuntime': runtime,
    })

________________________________________________________________________________________
Konstruktive Phase für: Falkenauer_u1000_13.json
Generating an initial solution according to BPI.
Constructive solution found: The number of bins is 1000. 

The allocation is feasible! All bins remain within their capacity.
Maximum weight in a bin: 100
Minimum weight in a bin: 20
{0: 100, 1: 100, 2: 100, 3: 100, 4: 100, 5: 100, 6: 100, 7: 100, 8: 100, 9: 99, 10: 99, 11: 99, 12: 99, 13: 99, 14: 99, 15: 99, 16: 99, 17: 99, 18: 99, 19: 98, 20: 98, 21: 98, 22: 98, 23: 98, 24: 98, 25: 98, 26: 98, 27: 98, 28: 98, 29: 98, 30: 98, 31: 97, 32: 97, 33: 97, 34: 97, 35: 97, 36: 96, 37: 96, 38: 96, 39: 96, 40: 96, 41: 96, 42: 96, 43: 96, 44: 96, 45: 95, 46: 95, 47: 95, 48: 95, 49: 95, 50: 95, 51: 95, 52: 95, 53: 95, 54: 95, 55: 95, 56: 95, 57: 95, 58: 95, 59: 95, 60: 95, 61: 94, 62: 94, 63: 94, 64: 94, 65: 94, 66: 94, 67: 94, 68: 94, 69: 94, 70: 94, 71: 94, 72: 94, 73: 94, 74: 94, 75: 94, 76: 94, 77: 94, 78: 94

# 4. Simulated Annealing

In [6]:
# Simulated Annealing sollte als Unterklasse von ImprovementAlgorithm in ImprovementAlgorithm.py umgesetzt werden.
# Solver.py ruft später algorithm.Initialize(...) und algorithm.Run(startSolution) auf.
# TODO: Nachbarschaften vollständig in SimulatedAnnealing/ImprovementAlgorithm über neighborhoodTypes erzeugen.
# TODO: Keine separaten Neighborhood-Objekte im Notebook initialisieren; Abschnitt 4 soll nur Parameter setzen und den Algorithmus starten.
# TODO: Sicherstellen, dass CreateNeighborhood mit der Lösungsstruktur arbeitet, die Solution tatsächlich bereitstellt.

finalResults = []

for result in constructiveResults:
    data = result['data']
    startSolution = result['startSolution']
    algorithm = SimulatedAnnealing(
        inputData=data,
        **SA_PARAMETERS,
    )
    solver = Solver(data)
    finalSolution = solver.Run(result['heuristic'], algorithm)

    finalSolution = startSolution
    result['finalSolution'] = finalSolution
    finalResults.append(result)

Generating an initial solution according to BPI.
Constructive solution found: The number of bins is 1000. 

Best found Solution: The number of bins is 490.

The allocation is feasible! All bins remain within their capacity.
Maximum weight in a bin: 150
Minimum weight in a bin: 22


# 5. Output und Ergebnisübersicht

In [7]:
def calculate_bin_weights(solution, data):
    binWeights = {binId: 0 for binId in sorted(set(solution.Allocation.values()))}

    for itemId, binId in solution.Allocation.items():
        binWeights[binId] += data.InputItems[itemId].weight

    return binWeights


def is_feasible(solution, data):
    binWeights = calculate_bin_weights(solution, data)
    capacity = data.InputBinCapacity.capacity
    return all(weight <= capacity for weight in binWeights.values())


def write_solution_csv(solution, data, outputFolder='../Solutions'):
    os.makedirs(outputFolder, exist_ok=True)

    instanceName = os.path.splitext(data.filename)[0]
    outputPath = os.path.join(outputFolder, f'Solution-{instanceName}.csv')
    binIdMap = {binId: index for index, binId in enumerate(sorted(set(solution.Allocation.values())))}

    with open(outputPath, 'w', newline='') as outputFile:
        writer = csv.writer(outputFile)
        writer.writerow(['itemId', 'binId'])

        for itemId in sorted(solution.Allocation):
            writer.writerow([itemId, binIdMap[solution.Allocation[itemId]]])

    return outputPath

results = []

for result in finalResults:
    data = result['data']
    finalSolution = result['finalSolution']
    finalSolution.FeasibilityCheck(data)
    feasible = is_feasible(finalSolution, data)
    solutionPath = write_solution_csv(finalSolution, data) if feasible else None

    if solutionPath is not None:
        print(f"Lösung gespeichert: {solutionPath}")
    else:
        print(f"Keine CSV geschrieben, da die Lösung für {result['instance']} nicht zulässig ist.")

    results.append({
        'Instanz': result['instance'],
        'Heuristik': result['heuristic'],
        'StartBins': result['startSolution'].NumberOfBins,
        'FinalBins': finalSolution.NumberOfBins,
        'Konstruktionszeit': round(result['constructiveRuntime'], 4),
        'Machbar': feasible,
        'SolutionFile': solutionPath,
    })

columns = ['Instanz', 'Heuristik', 'StartBins', 'FinalBins', 'Konstruktionszeit', 'Machbar', 'SolutionFile']
columnWidths = {
    column: max(len(column), *(len(str(row[column])) for row in results))
    for column in columns
}

header = ' | '.join(column.ljust(columnWidths[column]) for column in columns)
separator = '-+-'.join('-' * columnWidths[column] for column in columns)
print(header)
print(separator)

for row in results:
    print(' | '.join(str(row[column]).ljust(columnWidths[column]) for column in columns))

results

The allocation is feasible! All bins remain within their capacity.
Maximum weight in a bin: 100
Minimum weight in a bin: 20
Lösung gespeichert: ../Solutions\Solution-Falkenauer_u1000_13.csv
Instanz                  | Heuristik | StartBins | FinalBins | Konstruktionszeit | Machbar | SolutionFile                                 
-------------------------+-----------+-----------+-----------+-------------------+---------+----------------------------------------------
Falkenauer_u1000_13.json | BPI       | 1000      | 1000      | 0.001             | True    | ../Solutions\Solution-Falkenauer_u1000_13.csv


[{'Instanz': 'Falkenauer_u1000_13.json',
  'Heuristik': 'BPI',
  'StartBins': 1000,
  'FinalBins': 1000,
  'Konstruktionszeit': 0.001,
  'Machbar': True,
  'SolutionFile': '../Solutions\\Solution-Falkenauer_u1000_13.csv'}]

## Optionaler Grid Search zur Parameterwahl

Dieser Block dient dazu, die Parameter von Simulated Annealing systematisch zu vergleichen, statt sie nur manuell zu setzen. Dafür werden mehrere Kombinationen aus Starttemperatur und Abkühlgeschwindigkeit getestet und anhand von Lösungsqualität und Laufzeit bewertet. Die Auswertung erzeugt Empfehlungen für unterschiedliche Ziele: beste Qualität, kürzeste Laufzeit, ausgewogene Laufzeit-Qualitäts-Balance und schnellste noch akzeptable Lösung.

Der Grid Search ist standardmäßig deaktiviert, weil er je nach Instanzgröße viele SA-Läufe ausführt. Besonders Laufzeiten und daraus abgeleitete Scores sind hardwareabhängig: Auf einem anderen Rechner können absolute Zeiten und damit auch Laufzeit-Qualitäts-Verhältnisse anders ausfallen, während die reine Lösungsqualität besser vergleichbar bleibt.


In [ ]:
from copy import deepcopy
from itertools import product

try:
    from sklearn.model_selection import KFold, ParameterGrid
except ImportError:
    KFold = None
    ParameterGrid = None

DO_OPTIMIZATION = False
GRID_SEARCH_OUTPUT_FOLDER = '../Figures'

SA_PARAMETER_GRID = {
    'temperature': [0.75, 0.9, 0.95],
    'coolingSpeed': [0.1, 0.25, 0.5, 0.75,],
    'neighborhoodEvaluationStrategy': ['FirstImprovement'],
    'neighborhoodTypes': [['']],
}
#TODO Extend grid search for neighborhood size 

def iter_parameter_grid(parameterGrid):
    if ParameterGrid is not None:
        yield from ParameterGrid(parameterGrid)
        return

    keys = list(parameterGrid)
    for values in product(*(parameterGrid[key] for key in keys)):
        yield dict(zip(keys, values))


def make_folds(items, n_splits=2):
    indices = np.arange(len(items))

    if len(indices) < 2:
        return [(indices, indices)]

    if KFold is not None:
        splitter = KFold(n_splits=min(n_splits, len(indices)), shuffle=True, random_state=seed)
        return list(splitter.split(indices))

    shuffled = np.random.default_rng(seed).permutation(indices)
    splitPoint = max(1, len(shuffled) // 2)
    return [
        (shuffled[:splitPoint], shuffled[splitPoint:]),
        (shuffled[splitPoint:], shuffled[:splitPoint]),
    ]


def run_sa_once(result, parameters, runSeed):
    data = result['data']
    evaluationLogic = EvaluationLogic(data)
    solutionPool = SolutionPool()
    startSolution = deepcopy(result['startSolution'])
    solutionPool.AddSolution(startSolution)

    algorithm = SimulatedAnnealing(inputData=data, **parameters)
    algorithm.Initialize(evaluationLogic, solutionPool, np.random.default_rng(runSeed))

    startTime = time.time()
    finalSolution = algorithm.Run(startSolution)
    runtime = time.time() - startTime

    evaluationLogic.CalculateNumberOfBins(finalSolution)
    return {
        'instance': result['instance'],
        'startBins': result['startSolution'].NumberOfBins,
        'finalBins': finalSolution.NumberOfBins,
        'improvement': result['startSolution'].NumberOfBins - finalSolution.NumberOfBins,
        'runtime': runtime,
    }


def add_grid_search_scores(rows):
    if not rows:
        return rows

    bestBins = min(row['meanFinalBins'] for row in rows)
    fastestRuntime = min(row['meanRuntime'] for row in rows)
    slowestRuntime = max(row['meanRuntime'] for row in rows)
    worstBins = max(row['meanFinalBins'] for row in rows)
    runtimeRange = max(slowestRuntime - fastestRuntime, 1e-12)
    qualityRange = max(worstBins - bestBins, 1e-12)

    for row in rows:
        qualityPenalty = (row['meanFinalBins'] - bestBins) / qualityRange
        runtimePenalty = (row['meanRuntime'] - fastestRuntime) / runtimeRange
        row['runtimeQualityRatio'] = row['meanRuntime'] / max(row['meanImprovement'], 1e-9)
        row['squaredRuntimeQualityRatio'] = (row['meanRuntime'] ** 2) / max(row['meanImprovement'], 1e-9)
        row['balancedScore'] = qualityPenalty + runtimePenalty

    sortedByQuality = sorted(rows, key=lambda row: (row['meanFinalBins'], row['meanRuntime']))
    bestQuality = sortedByQuality[0]
    acceptableQualityLimit = bestQuality['meanFinalBins'] * 1.02
    acceptableRows = [row for row in rows if row['meanFinalBins'] <= acceptableQualityLimit]

    return {
        'allResults': rows,
        'bestQuality': bestQuality,
        'bestRuntime': min(rows, key=lambda row: row['meanRuntime']),
        'bestBalanced': min(rows, key=lambda row: row['balancedScore']),
        'bestRuntimeQualityRatio': min(rows, key=lambda row: row['runtimeQualityRatio']),
        'bestSquaredRuntimeQualityRatio': min(rows, key=lambda row: row['squaredRuntimeQualityRatio']),
        'fastestAcceptable': min(acceptableRows, key=lambda row: row['meanRuntime']),
    }


def plot_grid_search_results(gridSearchResults, outputFolder):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('matplotlib ist nicht installiert; Grid-Search-Plots werden übersprungen.')
        return []

    os.makedirs(outputFolder, exist_ok=True)
    rows = gridSearchResults['allResults']
    plotPaths = []

    def parameter_label(row):
        return f"T={row['temperature']}, C={row['coolingSpeed']}"

    def plot_bar_chart(filename, metricKey, title, yLabel):
        sortedRows = sorted(rows, key=lambda row: row[metricKey])
        labels = [parameter_label(row) for row in sortedRows]
        values = [row[metricKey] for row in sortedRows]
        xPositions = range(len(sortedRows))
        figureWidth = max(9, len(sortedRows) * 0.75)

        path = os.path.join(outputFolder, filename)
        plt.figure(figsize=(figureWidth, 4.5))
        plt.bar(xPositions, values)
        plt.xticks(xPositions, labels, rotation=45, ha='right')
        plt.xlabel('Gridsearch-Parameterpaar')
        plt.ylabel(yLabel)
        plt.title(title)
        plt.tight_layout()
        plt.savefig(path, dpi=150)
        plt.close()
        plotPaths.append(path)

    for row in rows:
        row['runtimeRelativeToQuality'] = row['meanRuntime'] * row['meanFinalBins']

    plotDefinitions = [
        ('gridsearch-runtime.png', 'meanRuntime', 'Gridsearch sortiert nach Laufzeit', 'Mittlere Laufzeit (s)'),
        ('gridsearch-quality-makespan.png', 'meanFinalBins', 'Gridsearch sortiert nach Ergebnisqualität', 'Mittlerer Makespan / FinalBins'),
        ('gridsearch-runtime-relative-quality.png', 'runtimeRelativeToQuality', 'Gridsearch sortiert nach Laufzeit relativ zur Qualität', 'Mittlere Laufzeit × Makespan'),
    ]

    for filename, metricKey, title, yLabel in plotDefinitions:
        plot_bar_chart(filename, metricKey, title, yLabel)

    return plotPaths


gridSearchResults = None

if DO_OPTIMIZATION:
    rawGridSearchResults = []
    folds = make_folds(constructiveResults, n_splits=2)

    for parameterIndex, parameters in enumerate(iter_parameter_grid(SA_PARAMETER_GRID), start=1):
        foldMetrics = []

        for foldIndex, (_, validationIndices) in enumerate(folds, start=1):
            for validationIndex in validationIndices:
                runSeed = seed + parameterIndex * 1000 + foldIndex * 100 + int(validationIndex)
                foldMetrics.append(run_sa_once(constructiveResults[int(validationIndex)], parameters, runSeed))

        rawGridSearchResults.append({
            **parameters,
            'meanFinalBins': float(np.mean([metric['finalBins'] for metric in foldMetrics])),
            'meanImprovement': float(np.mean([metric['improvement'] for metric in foldMetrics])),
            'meanRuntime': float(np.mean([metric['runtime'] for metric in foldMetrics])),
            'foldMetrics': foldMetrics,
        })

    gridSearchResults = add_grid_search_scores(rawGridSearchResults)
    gridSearchPlotPaths = plot_grid_search_results(gridSearchResults, GRID_SEARCH_OUTPUT_FOLDER)

    BEST_QUALITY_PARAMETERS = {key: gridSearchResults['bestQuality'][key] for key in SA_PARAMETERS}
    BEST_RUNTIME_PARAMETERS = {key: gridSearchResults['bestRuntime'][key] for key in SA_PARAMETERS}
    BEST_BALANCED_PARAMETERS = {key: gridSearchResults['bestBalanced'][key] for key in SA_PARAMETERS}
    BEST_FAST_ACCEPTABLE_PARAMETERS = {key: gridSearchResults['fastestAcceptable'][key] for key in SA_PARAMETERS}

    SA_PARAMETERS = BEST_BALANCED_PARAMETERS

    print('Beste Qualitätsparameter:', BEST_QUALITY_PARAMETERS)
    print('Schnellste Parameter:', BEST_RUNTIME_PARAMETERS)
    print('Ausgewogene Parameter:', BEST_BALANCED_PARAMETERS)
    print('Schnellste akzeptable Parameter:', BEST_FAST_ACCEPTABLE_PARAMETERS)
    print('Gespeicherte Plots:', gridSearchPlotPaths)
else:
    print('Grid Search ist deaktiviert. Setze DO_OPTIMIZATION = True, um SA_PARAMETERS automatisch zu kalibrieren.')

Beste Qualitätsparameter: {'neighborhoodEvaluationStrategy': 'FirstImprovement', 'neighborhoodTypes': ['Swap'], 'temperature': 0.9, 'coolingSpeed': 0.75}
Schnellste Parameter: {'neighborhoodEvaluationStrategy': 'FirstImprovement', 'neighborhoodTypes': ['Swap'], 'temperature': 0.9, 'coolingSpeed': 0.1}
Ausgewogene Parameter: {'neighborhoodEvaluationStrategy': 'FirstImprovement', 'neighborhoodTypes': ['Swap'], 'temperature': 0.9, 'coolingSpeed': 0.25}
Schnellste akzeptable Parameter: {'neighborhoodEvaluationStrategy': 'FirstImprovement', 'neighborhoodTypes': ['Swap'], 'temperature': 0.95, 'coolingSpeed': 0.75}
Gespeicherte Plots: ['../Figures\\gridsearch-runtime.png', '../Figures\\gridsearch-quality-makespan.png', '../Figures\\gridsearch-runtime-relative-quality.png']


# 6. Dokumentationsnotizen

In [9]:
documentationChecklist = [
    'Problemklassifikation: Bin Packing als kombinatorisches Optimierungsproblem',
    'Klassifizierung: Simulated Annealing als trajektorienbasierte, stochastische Metaheuristik',
    'Suchoperatoren: MoveItem und optional SwapItems aus ImprovementAlgorithm.py',
    'Diversifizierung: Akzeptanz schlechterer Lösungen bei hoher Temperatur',
    'Intensivierung: sinkende Temperatur fokussiert die Suche auf Verbesserungen',
    'Terminierungskriterium: Temperatur, Iterationslimit und Stagnation',
    'Parameterwahl und kurze Begründung',
    'Implementierungsvorgang und Schwierigkeiten',
    'Lösungen aller Datensätze als CSV-Output',
    'Wortumfang zwischen 500 und 2000 Wörtern',
]

# TODO: Diese Punkte als Markdown-Text im Notebook ausformulieren.
# TODO: Pseudocode Dokumentation:
# TODO:   1. Problem und Zielfunktion erklären.
# TODO:   2. Konstruktive Heuristik aus ConstructiveHeuristics.py beschreiben.
# TODO:   3. Metaheuristik aus ImprovementAlgorithm.py beschreiben.
# TODO:   4. Ergebnisse aus results interpretieren.
# TODO:   5. Schwierigkeiten und Parameterentscheidungen begründen.

documentationChecklist

['Problemklassifikation: Bin Packing als kombinatorisches Optimierungsproblem',
 'Klassifizierung: Simulated Annealing als trajektorienbasierte, stochastische Metaheuristik',
 'Suchoperatoren: MoveItem und optional SwapItems aus ImprovementAlgorithm.py',
 'Diversifizierung: Akzeptanz schlechterer Lösungen bei hoher Temperatur',
 'Intensivierung: sinkende Temperatur fokussiert die Suche auf Verbesserungen',
 'Terminierungskriterium: Temperatur, Iterationslimit und Stagnation',
 'Parameterwahl und kurze Begründung',
 'Implementierungsvorgang und Schwierigkeiten',
 'Lösungen aller Datensätze als CSV-Output',
 'Wortumfang zwischen 500 und 2000 Wörtern']